# Module 7 — Saving results, figures and caching

**Who this is for:** anyone running more than a handful of experiments, and anyone whose pipeline is slow to re-run.

> **Self-check — skip this module if you can already:**
>
> - [ ] save each run's settings, metrics and predictions so you can find them again next week
> - [ ] save a figure at a quality fit for the report
> - [ ] cache a slow preprocessing or feature step so it only runs again when its settings change
> - [ ] explain what MLflow or Weights & Biases add on top of saving files
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- one folder per run: config, metrics, predictions, figures
- figures for reports
- caching preprocessing and features with `joblib.Memory` or a config hash
- what to cache and what not to
- optional, advanced: experiment tracking with MLflow and W&B

**Where the course uses this:** `RESULTS.md` (rubric Criterion 9), the report's figures, and every slow `preprocess()` / `extract_features()` run in the capstone.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/07_results_figures_caching.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

In [ ]:
import json, time, hashlib, os, sys, platform, shlex, subprocess, tempfile, uuid
from datetime import datetime
from pathlib import Path
import importlib.metadata as im
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import trapezoid      # area under a curve (band power)

WORK = Path(tempfile.mkdtemp())   # a throw-away folder for this tutorial. In your project use folders that
print(WORK)                       # stay: Path("results"), Path("cache") next to your code.

This module has two independent examples, and each gets its **own** configuration, so that every saved
setting really did affect the result saved next to it:

- `eval_cfg` — a small classification study whose results we save (section 1);
- `feature_cfg` — signal-processing settings for the caching example (section 3).

## 1. One folder per run

Every time you run an experiment, write everything it produced into its own folder, named by date, time and
a short description. Then "which settings gave kappa 0.62?" is answered by opening a folder, not by memory.

```
results/
  2026-10-15_141203_logreg-baseline_3fa9c1/
    config.json        # every setting that produced this result, including the seed
    model_params.json  # the estimator's actual parameters (model.get_params())
    metrics.json       # the numbers you report, computed from predictions.csv
    predictions.csv    # one row per epoch: subject, epoch, fold, true label, predicted label
    environment.txt    # Python, platform, code revision and every installed library version
    *.pdf / *.png      # figures
```

The short random suffix makes it very unlikely that two runs started in the same second collide. Below, a
small real evaluation produces the predictions, the metrics are **computed from them**, and the config lists
exactly the settings that produced them, so the folder is self-consistent.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score

eval_cfg = {"seed": 2013, "data": "synthetic-eval-v1", "n_subjects": 8, "n_epochs_per_subject": 30,
            "n_features": 4, "class_effect": 1.5, "subject_offset_sd": 1.0, "n_folds": 4,
            "model": "StandardScaler + LogisticRegression(max_iter=1000)"}

def new_run_dir(root, name):
    stamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    d = Path(root) / f"{stamp}_{name}_{uuid.uuid4().hex[:6]}"
    d.mkdir(parents=True)
    return d

# the synthetic study, generated ONLY from eval_cfg
c = eval_cfg
rng = np.random.default_rng(c["seed"])
subject = np.repeat(np.arange(c["n_subjects"]), c["n_epochs_per_subject"])
epoch = np.tile(np.arange(c["n_epochs_per_subject"]), c["n_subjects"])
y = rng.integers(0, 3, subject.size)
X = (rng.standard_normal((subject.size, c["n_features"])) + c["class_effect"] * y[:, None]
     + rng.normal(0, c["subject_offset_sd"], c["n_subjects"])[subject][:, None])

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pred, fold, fold_kappa = np.empty_like(y), np.empty_like(y), []
for k, (tr, te) in enumerate(GroupKFold(c["n_folds"]).split(X, y, subject)):
    model.fit(X[tr], y[tr])
    pred[te], fold[te] = model.predict(X[te]), k
    fold_kappa.append(cohen_kappa_score(y[te], pred[te]))

run = new_run_dir(WORK / "results", "logreg-baseline")
(run / "config.json").write_text(json.dumps(eval_cfg, indent=2))
# the estimator's actual parameters too, so the description in eval_cfg can never drift from the real model
(run / "model_params.json").write_text(json.dumps(model.get_params(deep=True), indent=2, default=str))
np.savetxt(run / "predictions.csv", np.c_[subject, epoch, fold, y, pred], fmt="%d", delimiter=",",
           header="subject,epoch,fold,true,pred", comments="")

metrics = {"pooled_oof_kappa": round(cohen_kappa_score(y, pred), 4),
           "pooled_oof_macro_f1": round(f1_score(y, pred, average="macro"), 4),
           "fold_kappa_mean": round(float(np.mean(fold_kappa)), 4),
           "fold_kappa_sd": round(float(np.std(fold_kappa)), 4), "n_folds": c["n_folds"]}
(run / "metrics.json").write_text(json.dumps(metrics, indent=2))

try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    commit = ""
env = [f"python {sys.version.split()[0]}", f"platform {platform.platform()}", f"git commit {commit or 'unknown'}"]
env += sorted(f"{d.metadata['Name']}=={d.version}" for d in im.distributions())
(run / "environment.txt").write_text("\n".join(env))
print(run.name, sorted(p.name for p in run.iterdir()))
print(json.dumps(metrics))

**Check that the saved files reproduce the reported numbers.** Anyone (including you, next month) can then
recompute any metric that depends only on labels, per fold or pooled:

In [ ]:
saved = np.loadtxt(run / "predictions.csv", delimiter=",", skiprows=1, dtype=int)
s_subject, s_epoch, s_fold, s_true, s_pred = saved.T
assert np.isclose(cohen_kappa_score(s_true, s_pred), json.loads((run / "metrics.json").read_text())["pooled_oof_kappa"], atol=1e-4)
print("per-fold kappa from the file:",
      [round(cohen_kappa_score(s_true[s_fold == k], s_pred[s_fold == k]), 3) for k in range(eval_cfg["n_folds"])])

If you report a metric that needs **scores** rather than labels (AUC, calibration), save the predicted
probabilities too (`model.predict_proba`).

Add one row to `RESULTS.md` per run you keep, pointing at its folder, and say what changed and why.
**Keep the `results/` folder out of git** if it holds large files; commit `RESULTS.md` and the small JSON files
if your team wants them versioned.

## 2. Figures that work in a report

A figure is read at the size it is printed. Most bad figures are fine on screen and unreadable on the page:
tiny fonts, blurry pixels, colours that disappear in print. Decide the **final size first**, then make the
figure at that size.

| Choice | Good default | Why |
|---|---|---|
| Size | width **3.5 in** (one column) or **7 in** (full page); height about 0.6 × width | fonts then appear at the size you set |
| Fonts | 8–10 pt at final size, one sans-serif family (e.g. DejaVu Sans, Arial); same size in every figure | readable, consistent |
| Format for the report | **PDF or SVG** (vector) | sharp at any zoom; text stays text |
| Format for slides / web | **PNG at 300 dpi** (print) or 150 dpi (screen) | raster images need enough pixels |
| Colour | `tab10` for categories, `viridis` / `cividis` for continuous values; avoid red–green pairs and `jet` | readable for colour-blind readers and in greyscale |
| Lines | 1–1.5 pt, distinguish by style (`-`, `--`, `:`) as well as colour | survives black-and-white printing |
| Axes | label every axis **with units**; use a log scale for spectra; same limits when panels are compared | honest, comparable |
| Panels | label them (a), (b), (c) and refer to them in the caption | the text can point at them |
| Avoid | 3-D bars, pie charts, rainbow colour maps, titles that repeat the caption | they hide the data |

Set the style **once** at the top of your notebook or project, so every figure matches:

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "figure.dpi": 110,                  # on-screen preview size
    "savefig.dpi": 300,                 # resolution of saved PNGs
    "font.family": "sans-serif",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 1.2,
    "pdf.fonttype": 42,                 # embed real fonts in PDFs (editable text, no font errors)
    "svg.fonttype": "none",
})

In [ ]:
fs_fig = 100
t = np.arange(10 * fs_fig) / fs_fig
x = np.sin(2 * np.pi * 1.2 * t) + 0.5 * np.sin(2 * np.pi * 10 * t) + 0.2 * rng.standard_normal(t.size)
sos = signal.butter(4, 5, btype="lowpass", fs=fs_fig, output="sos")
x_f = signal.sosfiltfilt(sos, x)
f, P = signal.welch(x, fs=fs_fig, nperseg=4 * fs_fig)
f, P_f = signal.welch(x_f, fs=fs_fig, nperseg=4 * fs_fig)

fig, ax = plt.subplots(1, 2, figsize=(7, 2.4))            # full page width, decided up front
ax[0].plot(t, x, color="0.6", lw=0.8, label="raw")
ax[0].plot(t, x_f, color="C0", label="low-pass 5 Hz")
ax[0].set(xlim=(0, 4), xlabel="time (s)", ylabel="amplitude (mV)")
ax[0].legend(frameon=False, loc="upper right")
ax[1].semilogy(f, P, color="0.6", lw=0.8, label="raw")
ax[1].semilogy(f, P_f, color="C0", ls="--", label="low-pass 5 Hz")
ax[1].set(xlim=(0, 30), xlabel="frequency (Hz)", ylabel="PSD (mV²/Hz)")
for a_, lab in zip(ax, "ab"):
    a_.text(-0.18, 1.02, f"({lab})", transform=a_.transAxes, fontweight="bold")
fig.tight_layout()
fig.savefig(run / "filter_example.pdf", bbox_inches="tight")   # for the report
fig.savefig(run / "filter_example.png", bbox_inches="tight")   # 300 dpi from rcParams, for slides
plt.show()
print([p.name for p in run.glob("filter_example.*")])

**A quick test before you hand in:** open the PDF of your report at 100 % zoom (or print one page). Can you
read every axis label without zooming? Can you tell the lines apart in greyscale? If not, fix the figure, not
the caption.

Two more habits:

- **Generate every figure from a script or notebook cell** that reads saved results, so changing the data
  regenerates the figure. Never edit figures by hand.
- **In loops, close figures** with `plt.close(fig)` after saving, or memory fills up.

## 3. Caching slow steps

Loading and preprocessing a full dataset can take minutes; extracting features from thousands of epochs
can take longer. If you only change the classifier, there is no reason to redo those steps.

**The rule:** a cached result may only be reused for *exactly* the inputs that produced it: the data, every
relevant setting, the seed if anything is random, and the **code** that computed it. If any of these can
change without the cache noticing, you will eventually get stale results without any error.

The settings for this example:

In [ ]:
feature_cfg = {"seed": 2013, "fs": 100, "epoch_s": 30, "bandpass_hz": [0.5, 35.0], "welch_nperseg_s": 4,
               "classifier": "logistic_regression"}     # the classifier does NOT affect the features
DATA_VERSION = "synthetic-v1"   # with real data: e.g. a checksum of the raw files (see the track manifests)
PIPELINE_VERSION = 1            # bump whenever you change ANY code that affects the cached results:
                                # loading, seed derivation, preprocessing or feature extraction

### Option A: `joblib.Memory` (installed with scikit-learn)

Decorate a function; joblib stores the result on disk, keyed by the function's code **and** its arguments.
Call it again with the same arguments and it loads the stored result instead of recomputing. It does **not**
notice if a data file the function reads has changed, so pass something that identifies the data (here
`DATA_VERSION`) as an argument, or clear the cache with `memory.clear()`.

In [ ]:
from joblib import Memory
memory = Memory(WORK / "cache", verbose=0)

def subject_seed(seed, subject):
    """A stable per-subject seed. Python's built-in hash() of a string changes every session:
    never use it as a seed."""
    payload = f"{seed}:{subject}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")

@memory.cache
def preprocess(subject, seed, fs, band, data_version, pipeline_version):
    time.sleep(1.0)                                   # stands in for a slow load + filter
    raw = np.random.default_rng(subject_seed(seed, subject)).standard_normal(120 * fs)   # "the recording"
    sos = signal.butter(4, band, btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, raw)

fc = feature_cfg
args = ("S01", fc["seed"], fc["fs"], tuple(fc["bandpass_hz"]), DATA_VERSION, PIPELINE_VERSION)
for attempt in (1, 2):
    t0 = time.perf_counter()
    out = preprocess(*args)
    print(f"call {attempt}: {time.perf_counter() - t0:.2f} s")

t0 = time.perf_counter()
preprocess("S01", fc["seed"], fc["fs"], (1.0, 30.0), DATA_VERSION, PIPELINE_VERSION)   # new band: recomputed
print(f"new band: {time.perf_counter() - t0:.2f} s")

### Option B: files named by a hash of everything that went into them

When you want plain files you can look at, name each one with a hash of **all** its inputs, and store those
inputs next to the arrays. Change any of them and the name changes, so the old file is not reused. Settings
that don't affect the features (here, the classifier) are left out of the key on purpose.

In [ ]:
def key_of(inputs):
    return hashlib.sha256(json.dumps(inputs, sort_keys=True).encode()).hexdigest()[:12]

def feature_inputs(subjects, cfg):
    return {"subjects": list(subjects), "data_version": DATA_VERSION, "pipeline_version": PIPELINE_VERSION,
            **{k: cfg[k] for k in ("seed", "fs", "epoch_s", "bandpass_hz", "welch_nperseg_s")}}

def feature_path(subjects, cfg, cache_dir):
    return Path(cache_dir) / f"features_{key_of(feature_inputs(subjects, cfg))}.npz"

def compute_features(subjects, cfg):
    fs, w = cfg["fs"], cfg["epoch_s"] * cfg["fs"]
    rows, grp = [], []
    for s in subjects:
        x = preprocess(s, cfg["seed"], fs, tuple(cfg["bandpass_hz"]), DATA_VERSION, PIPELINE_VERSION)
        for k in range(x.size // w):
            f, P = signal.welch(x[k * w:(k + 1) * w], fs=fs, nperseg=cfg["welch_nperseg_s"] * fs)
            rows.append([np.log(trapezoid(P[(f >= lo) & (f <= hi)], f[(f >= lo) & (f <= hi)]))
                         for lo, hi in [(1, 4), (4, 8), (8, 12)]])
            grp.append(s)
    return np.array(rows), np.array(grp)

def features_for(subjects, cfg, cache_dir):
    path = feature_path(subjects, cfg, cache_dir)
    if path.exists():
        with np.load(path) as d:
            print("cache hit ", path.name)
            return d["X"], d["groups"]
    X, grp = compute_features(subjects, cfg)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, X=X, groups=grp, inputs=json.dumps(feature_inputs(subjects, cfg)))  # provenance
    print("cache miss", path.name)
    return X, grp

Now **test the cache**: the same inputs must give the same result even with every cache empty, a relevant
change must miss, and an irrelevant change must hit.

In [ ]:
subjects = ["S01", "S02", "S03"]
X1, g1 = features_for(subjects, feature_cfg, WORK / "feat_a")              # miss: computed and saved
X1b, _ = features_for(subjects, feature_cfg, WORK / "feat_a")              # hit: loaded
assert np.array_equal(X1, X1b)

# Both cache layers empty: a new joblib cache AND a new feature folder must reproduce X1 exactly.
cached_preprocess = preprocess
preprocess = Memory(WORK / "cache_empty", verbose=0).cache(cached_preprocess.func)
X1c, _ = features_for(subjects, feature_cfg, WORK / "feat_empty")
preprocess = cached_preprocess
assert np.array_equal(X1, X1c), "same inputs must give the same features, even from empty caches"

features_for(subjects, {**feature_cfg, "seed": 999}, WORK / "feat_a")            # seed changed: must miss
features_for(subjects, {**feature_cfg, "classifier": "svm"}, WORK / "feat_a")    # classifier changed: must hit
print(sorted(p.name for p in (WORK / "feat_a").glob("*.npz")))

**The trap: changing code without telling the cache.** Suppose you improve `preprocess()` (here it now
doubles the signal) but forget to bump `PIPELINE_VERSION`. The feature file for these settings already
exists, so `features_for()` returns it **without ever calling** the new `preprocess()`: stale features, no
error. Bumping the version changes the key, and the features are recomputed:

In [ ]:
original_preprocess = preprocess

@memory.cache
def preprocess(subject, seed, fs, band, data_version, pipeline_version):     # "improved" code
    return 2 * original_preprocess.func(subject, seed, fs, band, data_version, pipeline_version)

X_stale, _ = features_for(subjects, feature_cfg, WORK / "feat_a")         # hit: the OLD features!
print("forgot to bump the version -> stale:", np.array_equal(X_stale, X1))

PIPELINE_VERSION = 2
X_new, _ = features_for(subjects, feature_cfg, WORK / "feat_a")           # miss: recomputed with the new code
print(f"after bumping: recomputed; log-power changed by {np.mean(X_new - X1):.3f} (= log 4, amplitude x2)")

preprocess, PIPELINE_VERSION = original_preprocess, 1                      # put the original back

### What to cache, and what to be careful with

| Safe to cache | Cache only with care | Don't cache |
|---|---|---|
| Downloaded raw data (`data_cache/` already does this) | Anything **fitted** on data (scalers, ICA, CSP, feature selection, models): only if the key includes the exact training subjects, settings and code. `Pipeline(memory=...)` in scikit-learn does this for fitted steps | Results of code you are still changing, unless you bump the version or clear the cache |
| Per-recording preprocessing and per-epoch features, computed from one recording at a time | | Anything that takes seconds to recompute |

Per-epoch features are safe because each one depends only on its own recording. A cached *fitted* step is
only a leak if it was fitted on data that includes your test set, or is reused for a different split.

One more subtlety: filtering a whole recording with `sosfiltfilt` before splitting is fine when you split **by
subject**. If you split **in time** within one recording (train on the first hour, test on the second), the
zero-phase filter uses samples from the test part, so use a causal filter or filter each part separately.
When in doubt, delete the cache folder: correctness first, speed second.

---

## 4. Advanced and optional: experiment tracking with MLflow and Weights & Biases

> **You do not need this for the course.** Saving one folder per run (section 1) is enough for the capstone.
> This section is for later: a thesis, a research project or a job, where you run hundreds of experiments and
> want to search, compare and plot them. Skip it if you are short of time.

**Experiment trackers** do what section 1 does by hand — store parameters, metrics, figures and files per run —
and add a searchable table and dashboards to compare runs.

| | MLflow | Weights & Biases (W&B) |
|---|---|---|
| Where data lives | your own disk (or a server you run) | W&B's cloud (an offline mode exists) |
| Account needed | no | yes, for the dashboard |
| Cost | open source, free | free tiers and an academic plan for eligible users; check the current terms |
| Install | `pip install mlflow` | `pip install wandb` |
| Look at results | `mlflow ui` in a terminal, then a local web page | the W&B website |

**Check before uploading any patient data or results derived from them to a cloud service**: the course
datasets are public, but in a real project data-protection rules apply.

The cells below run only if the library is installed, so the notebook still works without them.

In [ ]:
try:
    import mlflow
except ImportError:
    mlflow = None
    print("MLflow not installed: pip install mlflow  (optional)")

if mlflow:
    import logging
    logging.getLogger("mlflow").setLevel(logging.WARNING)            # keep the output short
    tracking_uri = "sqlite:///" + str(WORK / "mlflow.db")             # in a project: a file next to your code
    mlflow.set_tracking_uri(tracking_uri)
    name = "cm2013-capstone"
    if mlflow.get_experiment_by_name(name) is None:                   # keep artifacts next to the database
        mlflow.create_experiment(name, artifact_location=(WORK / "mlartifacts").as_uri())
    mlflow.set_experiment(name)
    with mlflow.start_run(run_name="logreg-baseline"):
        mlflow.log_params(eval_cfg)                                   # every setting of THIS run
        mlflow.log_metrics(metrics)                                   # every number you would report
        mlflow.log_artifact(str(run / "predictions.csv"))             # any file: predictions, figures, config
        mlflow.log_artifact(str(run / "filter_example.png"))
    runs = mlflow.search_runs(experiment_names=[name])
    print(runs[["tags.mlflow.runName", "params.seed", "metrics.pooled_oof_kappa"]])
    print("\nTo browse these runs, run this in a terminal:")
    print("mlflow ui --backend-store-uri " + shlex.quote(tracking_uri))

The printed command points at the exact database this notebook wrote, wherever that is. In your own project,
keep the database and artifacts in a folder that stays (not a temporary one), and run `mlflow ui` from there.

In [ ]:
try:
    import wandb
except ImportError:
    wandb = None
    print("W&B not installed: pip install wandb  (optional)")

if wandb:
    os.environ["WANDB_SILENT"] = "true"
    w = wandb.init(project="cm2013-capstone", config=eval_cfg, mode="offline", dir=str(WORK))
    wandb.log(metrics)                                   # can be called repeatedly, e.g. per fold or epoch
    w.finish()
    print("logged offline; `wandb sync` uploads it later if you have an account")

With an account you would use `mode="online"` (after `wandb login`), and the run appears on the W&B
website with charts. Offline mode, used here, needs no account.

**Whichever you use, the habits from sections 1–3 still apply:** a fixed seed, one config per run, and
results you can trace back to the code and settings that produced them. A tracker makes those easier to
search; it does not replace them.

## Try it yourself

Change the band-pass to `[1.0, 30.0]` (make a new config, `cfg2 = {**feature_cfg, "bandpass_hz": [1.0, 30.0]}`),
and compute its features into `X3` with `features_for(subjects, cfg2, WORK / "feat_a")`. It should be a cache
**miss**, give different features from `X1`, and leave the earlier files in place.

In [ ]:
# TODO

In [ ]:
# Check: keep this cell separate, so it survives when you paste in a solution.
try:
    new_file = feature_path(subjects, {**feature_cfg, "bandpass_hz": [1.0, 30.0]}, WORK / "feat_a")
    old_file = feature_path(subjects, feature_cfg, WORK / "feat_a")
    print("new features differ from X1:", "correct" if X3.shape == X1.shape and not np.array_equal(X3, X1) else "not yet")
    print("new cache file created:     ", "correct" if new_file.exists() else "not yet")
    print("earlier cache file kept:    ", "correct" if old_file.exists() else "not yet")
except NameError:
    print("Not done yet: compute X3.")

**Solution:** try it yourself first, then compare with the [solutions notebook](solutions.ipynb) (Module 7). Yours may differ and still be right.

**Next:** Module 8 — troubleshooting and common mistakes.